In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import shape
import json


pd.options.display.float_format = '{:.2f}'.format

Για να κάνεις spatial join με GeoPandas (και λίγο Pandas) σε γενικές γραμμές αν έχεις lat long στήλες πρέπει να:

Μετατρέψεις τα δεδομένα σου (CSV + GeoJSON) σε GeoDataFrames

Βεβαιωθείς ότι έχουν ίδιο CRS (σύστημα συντεταγμένων) -> Ελλάδα έχει διαφορετικό από όλη την Ευρώπη αλλά οκ

Χρησιμοποιήσεις sjoin για να δεις σε ποιο NUTS3 polygon πέφτει το καμένο polygon Ή overlay

In [2]:
df_geo = gpd.read_file("effis_burnt_areas_multipolygons_all_15-04-2026_merged_data.geojson")

C:\Users\kwnst\AppData\Local\Programs\Python\Python314\Lib\site-packages\pyogrio\raw.py:200: RuntimeWarning: Several features with id = 297038 have been found. Altering it to be unique. This warning will not be emitted anymore for this layer
  return ogr_read(


In [4]:
df_geo

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,...,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url,geometry,year
0,298163,PT,Portugal,Alto Minho,Chaviães e Paços,2026-04-06 15:42:20.036000+00:00,80,0.00000000,0.00000000,6.25000000,...,0.00000000,2026-04-14 13:42:28.703000+00:00,None,False,-8.22,42.12,"{'type': 'MultiPolygon', 'coordinates': [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,"MULTIPOLYGON (((-8.21082 42.12309, -8.21077 42...",2026
1,298161,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09 08:42:00+00:00,3,0.00000000,0.00000000,0.00000000,...,0.00000000,2026-04-14 13:34:38.376000+00:00,None,False,-8.55,41.96,"{'type': 'MultiPolygon', 'coordinates': [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,"MULTIPOLYGON (((-8.54835 41.9575, -8.54831 41....",2026
2,298162,PT,Portugal,Alto Minho,Insalde e Porreiras,2026-04-09 08:42:00+00:00,1,0.00000000,0.00000000,0.00000000,...,0.00000000,2026-04-14 13:34:38.376000+00:00,None,False,-8.55,41.96,"{'type': 'MultiPolygon', 'coordinates': [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,"MULTIPOLYGON (((-8.553 41.95475, -8.55295 41.9...",2026
3,298158,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-03-31 23:24:00+00:00,2,33.33333333,0.00000000,0.00000000,...,0.00000000,2026-04-14 13:27:50.324000+00:00,None,False,-8.59,41.95,"{'type': 'MultiPolygon', 'coordinates': [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,"MULTIPOLYGON (((-8.5929 41.95477, -8.59287 41....",2026
4,298159,PT,Portugal,Alto Minho,Formariz e Ferreira,2026-03-31 23:24:00+00:00,0,0.00000000,0.00000000,0.00000000,...,0.00000000,2026-04-14 13:27:50.324000+00:00,None,False,-8.59,41.95,"{'type': 'MultiPolygon', 'coordinates': [[[[-8...",https://api.effis.emergency.copernicus.eu/rest...,"MULTIPOLYGON (((-8.59374 41.95285, -8.59372 41...",2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116335,179103,HR,Hrvatska,N.A.,Draž,2000-11-08 23:00:00+00:00,17,0.00000000,0.00000000,0.00000000,...,0.00000000,2000-12-31 11:10:11.414000+00:00,None,False,18.73,45.89,"{'type': 'MultiPolygon', 'coordinates': [[[[18...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((18.729 45.88721, 18.7334 45.88...",2000
116336,179023,RO,România,Dolj,Negoi,2000-11-04 23:00:00+00:00,17,0.00000000,0.00000000,0.00000000,...,0.00000000,2000-12-31 11:10:11.414000+00:00,None,False,23.35,43.89,"{'type': 'MultiPolygon', 'coordinates': [[[[23...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((23.35645 43.8877, 23.35205 43....",2000
116337,179097,RO,România,Tulcea,C.A. Rosetti,2000-11-04 23:00:00+00:00,17,0.00000000,0.00000000,0.00000000,...,100.00000000,2000-12-31 11:10:11.414000+00:00,None,False,29.47,45.28,"{'type': 'MultiPolygon', 'coordinates': [[[[29...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((29.47363 45.28076, 29.46924 45...",2000
116338,258248,PT,Portugal,Douro,Urros e Peredo dos Castelhanos,2000-08-26 22:00:00+00:00,16,0.00000000,0.00000000,0.00000000,...,0.00000000,2000-12-31 11:10:11.414000+00:00,None,False,-6.98,41.10,"{'type': 'MultiPolygon', 'coordinates': [[[[-6...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((-6.98286 41.10105, -6.98445 41...",2000


## Keep EU countries and decade

In [6]:
df_geo["year"] = pd.to_datetime(df_geo["firedate"]).dt.year
df_decade = df_geo[(df_geo["year"] >= 2015) & (df_geo["year"] <= 2025)]
df_decade

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,...,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url,geometry,year
5854,288296,UK,United Kingdom,Gwynedd,Gwynedd,2025-12-30 10:34:00+00:00,6,0.00000000,0.00000000,0.00000000,...,0.00000000,2026-01-28 13:05:50.446000+00:00,None,True,-4.62,52.83,"{'type': 'MultiPolygon', 'coordinates': [[[[-4...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((-4.61904 52.83432, -4.61999 52...",2025
5857,288293,UA,Ukraine,N.A.,N.A.,2025-08-15 22:00:00+00:00,44,0.00000000,0.00000000,0.00000000,...,0.00000000,2026-01-26 14:00:12.299000+00:00,None,True,35.82,47.34,"{'type': 'MultiPolygon', 'coordinates': [[[[35...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((35.83254 47.34155, 35.83177 47...",2025
5858,288292,UA,Ukraine,N.A.,N.A.,2025-08-18 22:00:00+00:00,58,0.00000000,0.00000000,6.77966102,...,0.00000000,2026-01-26 13:08:31.081000+00:00,None,True,35.42,47.61,"{'type': 'MultiPolygon', 'coordinates': [[[[35...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((35.42449 47.60571, 35.424 47.6...",2025
5859,288291,UA,Ukraine,N.A.,N.A.,2025-08-18 22:00:00+00:00,18,0.00000000,0.00000000,11.76470588,...,0.00000000,2026-01-26 13:06:15.949000+00:00,None,True,35.40,47.59,"{'type': 'MultiPolygon', 'coordinates': [[[[35...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((35.39813 47.58826, 35.39961 47...",2025
5863,288287,UA,Ukraine,N.A.,N.A.,2025-08-13 22:00:00+00:00,89,0.00000000,0.00000000,0.00000000,...,0.00000000,2026-01-23 09:38:28.539000+00:00,None,True,29.99,47.27,"{'type': 'MultiPolygon', 'coordinates': [[[[30...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((30.00226 47.27391, 30.0002 47....",2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109556,74401,HR,Hrvatska,Splitsko-dalmatinska županija,Vrgorac,2022-01-19 23:00:00+00:00,16,73.33333333,0.00000000,0.00000000,...,0.00000000,2022-01-26 10:28:21.284000+00:00,None,False,17.20,43.32,"{'type': 'MultiPolygon', 'coordinates': [[[[17...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((17.20471 43.31999, 17.2043 43....",2022
109557,59406,BA,Bosnia and Herzegovina,Grude,N.A.,2022-01-19 23:28:00+00:00,7,0.00000000,0.00000000,0.00000000,...,0.00000000,2022-01-26 10:26:29.449000+00:00,None,True,17.28,43.34,"{'type': 'MultiPolygon', 'coordinates': [[[[17...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((17.28051 43.34626, 17.28321 43...",2022
109558,59405,HR,Hrvatska,Splitsko-dalmatinska županija,Runovići,2022-01-19 23:28:00+00:00,26,4.16666667,0.00000000,0.00000000,...,0.00000000,2022-01-26 10:24:50.822000+00:00,None,False,17.28,43.35,"{'type': 'MultiPolygon', 'coordinates': [[[[17...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((17.27843 43.34745, 17.27853 43...",2022
109559,59404,FR,France,Alpes-Maritimes,Thiéry,2022-01-20 10:53:00+00:00,5,0.00000000,0.00000000,0.00000000,...,0.00000000,2022-01-26 10:18:24.399000+00:00,None,False,7.05,43.98,"{'type': 'MultiPolygon', 'coordinates': [[[[7....",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((7.05622 43.97755, 7.05559 43.9...",2022


In [7]:
df_decade_sorted = df_decade.sort_values(by="firedate", ascending=False)
df_decade_sorted['country'].value_counts()

country
UA    22998
IT    10367
ES     7472
TR     6477
PT     6351
FR     5055
RO     4334
DZ     3807
UK     2431
BA     2348
AL     2172
RS     1858
ME     1776
EL     1629
LB     1209
SY     1137
MK     1067
BG     1039
KS      951
HR      886
TN      816
DE      605
MA      591
IE      490
SE      413
NO      389
IL      364
FI      282
PL      188
CY      168
HU      159
DK       99
LY       88
JO       56
LV       53
NL       41
BE       40
LT       37
PS       33
SI       30
EE       27
AT       25
SK       17
CZ       14
EG       13
CH        9
RU        8
MT        4
MD        2
GE        1
Name: count, dtype: int64

In [8]:
df_decade_eu = df_decade_sorted[df_decade_sorted["noneu"] == False]
df_decade_eu

,id,country,countryful,province,commune,firedate,area_ha,broadlea,conifer,mixed,...,percna2k,lastupdate,lastfiredate,noneu,lon,lat,shape_json,source_url,geometry,year
6062,288077,IT,Italia,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33333333,0.00000000,0.00000000,...,100.00000000,2026-01-07 12:26:03.984000+00:00,None,False,14.95,40.75,"{'type': 'MultiPolygon', 'coordinates': [[[[14...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((14.94778 40.74932, 14.94759 40...",2025
6063,288076,IT,Italia,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00000000,0.00000000,0.00000000,...,100.00000000,2026-01-07 12:25:40.928000+00:00,None,False,14.94,40.76,"{'type': 'MultiPolygon', 'coordinates': [[[[14...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((14.93568 40.75572, 14.93565 40...",2025
6031,288108,PT,Portugal,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,...,100.00000000,2026-01-07 13:42:42.695000+00:00,None,False,-8.38,41.92,"{'type': 'MultiPolygon', 'coordinates': [[[[-8...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((-8.38283 41.91932, -8.38284 41...",2025
6027,288112,FR,France,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00000000,0.00000000,0.00000000,...,0.00000000,2026-01-07 13:50:15.681000+00:00,None,False,2.60,45.06,"{'type': 'MultiPolygon', 'coordinates': [[[[2....",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((2.60406 45.06376, 2.60415 45.0...",2025
6044,288095,FR,France,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.66666667,0.00000000,0.00000000,...,0.00000000,2026-01-07 13:21:16.864000+00:00,None,False,2.61,45.02,"{'type': 'MultiPolygon', 'coordinates': [[[[2....",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((2.60793 45.02059, 2.60768 45.0...",2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,9109,PT,Portugal,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.11731844,0.00000000,0.00000000,...,99.46428403,2022-01-26 10:57:54.973000+00:00,None,False,-7.82,41.86,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((-7.82441 41.86385, -7.8275 41....",2015
92858,9195,PT,Portugal,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77358491,0.00000000,0.00000000,...,100.00000000,2022-01-26 10:57:54.973000+00:00,None,False,-7.80,41.87,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((-7.79748 41.86824, -7.79693 41...",2015
86634,10658,PT,Portugal,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00000000,0.00000000,0.00000000,...,100.00000000,2022-01-26 10:57:54.973000+00:00,None,False,-7.90,41.84,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((-7.90157 41.85033, -7.90371 41...",2015
92859,9363,PT,Portugal,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00000000,0.00000000,0.00000000,...,100.00000000,2022-01-26 10:57:54.973000+00:00,None,False,-7.84,41.85,"{'type': 'MultiPolygon', 'coordinates': [[[[-7...",http://api.effis.emergency.copernicus.eu/rest/...,"MULTIPOLYGON (((-7.84382 41.84751, -7.84421 41...",2015


### drop some columns to make it lighter 

In [9]:
df_smaller = df_decade_eu.drop(columns=[
    "countryful",
    "noneu",
    "lastfiredate",
    "lon",
    "lat",
    "shape_json",
    "source_url"
])
df_smaller

,id,country,province,commune,firedate,area_ha,broadlea,conifer,mixed,scleroph,transit,othernatlc,agriareas,artifsurf,otherlc,percna2k,lastupdate,geometry,year
6062,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33333333,0.00000000,0.00000000,0.00000000,66.66666666,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:26:03.984000+00:00,"MULTIPOLYGON (((14.94778 40.74932, 14.94759 40...",2025
6063,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:25:40.928000+00:00,"MULTIPOLYGON (((14.93568 40.75572, 14.93565 40...",2025
6031,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00000000,2026-01-07 13:42:42.695000+00:00,"MULTIPOLYGON (((-8.38283 41.91932, -8.38284 41...",2025
6027,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:50:15.681000+00:00,"MULTIPOLYGON (((2.60406 45.06376, 2.60415 45.0...",2025
6044,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.66666667,0.00000000,0.00000000,0.00000000,0.00000000,83.33333333,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:21:16.864000+00:00,"MULTIPOLYGON (((2.60793 45.02059, 2.60768 45.0...",2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.11731844,0.00000000,0.00000000,0.00000000,26.25698324,58.10055866,14.52513966,0.00000000,0.00000000,99.46428403,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.82441 41.86385, -7.8275 41....",2015
92858,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77358491,0.00000000,0.00000000,0.00000000,0.00000000,96.22641509,0.00000000,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.79748 41.86824, -7.79693 41...",2015
86634,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00000000,0.00000000,0.00000000,0.00000000,44.01913876,53.11004785,2.87081340,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.90157 41.85033, -7.90371 41...",2015
92859,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,86.79245283,13.20754717,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.84382 41.84751, -7.84421 41...",2015


## drop duplicates we want one fire per row

In [10]:
df_uniques = df_smaller.drop_duplicates(subset="id")
df_uniques

,id,country,province,commune,firedate,area_ha,broadlea,conifer,mixed,scleroph,transit,othernatlc,agriareas,artifsurf,otherlc,percna2k,lastupdate,geometry,year
6062,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33333333,0.00000000,0.00000000,0.00000000,66.66666666,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:26:03.984000+00:00,"MULTIPOLYGON (((14.94778 40.74932, 14.94759 40...",2025
6063,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:25:40.928000+00:00,"MULTIPOLYGON (((14.93568 40.75572, 14.93565 40...",2025
6031,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00000000,2026-01-07 13:42:42.695000+00:00,"MULTIPOLYGON (((-8.38283 41.91932, -8.38284 41...",2025
6027,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:50:15.681000+00:00,"MULTIPOLYGON (((2.60406 45.06376, 2.60415 45.0...",2025
6044,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.66666667,0.00000000,0.00000000,0.00000000,0.00000000,83.33333333,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:21:16.864000+00:00,"MULTIPOLYGON (((2.60793 45.02059, 2.60768 45.0...",2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.11731844,0.00000000,0.00000000,0.00000000,26.25698324,58.10055866,14.52513966,0.00000000,0.00000000,99.46428403,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.82441 41.86385, -7.8275 41....",2015
92858,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77358491,0.00000000,0.00000000,0.00000000,0.00000000,96.22641509,0.00000000,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.79748 41.86824, -7.79693 41...",2015
86634,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00000000,0.00000000,0.00000000,0.00000000,44.01913876,53.11004785,2.87081340,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.90157 41.85033, -7.90371 41...",2015
92859,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,86.79245283,13.20754717,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.84382 41.84751, -7.84421 41...",2015


In [11]:
df_uniques.sort_values(by="area_ha", ascending=False)

,id,country,province,commune,firedate,area_ha,broadlea,conifer,mixed,scleroph,transit,othernatlc,agriareas,artifsurf,otherlc,percna2k,lastupdate,geometry,year
55669,218736,EL,Έβρος,Τοπική Κοινότητα Δωρικού,2023-08-19 06:39:00+00:00,96610,17.59237212,1.61398859,28.35091570,15.91833777,13.56827099,4.26946052,17.57270196,0.74850144,0.36545091,64.68081331,2023-09-13 12:12:20.726000+00:00,"MULTIPOLYGON (((26.16024 41.21321, 26.16038 41...",2023
82068,10170,PT,Viseu Dão Lafões,Pinheiro de Ázere,2017-10-13 22:00:00+00:00,67521,12.15912119,7.35054629,32.63405679,0.00000000,20.39054866,0.00000000,22.50466349,1.90388772,3.05717585,1.04522268,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-8.013 40.56418, -8.01358 40.5...",2017
82069,10293,PT,Região de Coimbra,Lagos da Beira e Lajeosa,2017-10-14 22:00:00+00:00,64321,1.56721291,19.82337759,9.37684629,0.00000000,32.56475637,6.12270282,29.39923505,1.14586896,0.00000000,15.04806286,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.78764 40.57132, -7.78739 40...",2017
15041,277926,PT,Beiras e Serra da Estrela,Trancoso (São Pedro e Santa Maria) e Souto Maior,2025-08-10 00:00:00+00:00,62104,0.81111093,5.15312938,0.81111093,0.36532179,29.65865748,31.13121007,31.89725928,0.12874777,0.04345237,0.00000000,2025-08-12 05:28:20.714000+00:00,"MULTIPOLYGON (((-7.48906 40.91232, -7.48883 40...",2025
60310,213578,EL,Εύβοια,Τοπική Κοινότητα Κουρκουλών,2021-08-03 09:27:23.570000+00:00,51881,4.82807802,28.60419397,19.72477064,2.52871791,9.60026212,0.20815666,33.82352941,0.67458176,0.00770951,1.61524333,2023-03-14 10:44:55.282000+00:00,"MULTIPOLYGON (((23.31446 39.03657, 23.31183 39...",2021
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11584,281948,PT,Ave,Infias,2025-09-21 11:36:00+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000000,2025-09-24 12:49:28.304000+00:00,"MULTIPOLYGON (((-8.31373 41.39978, -8.31374 41...",2025
38108,242889,PT,Região de Aveiro,Pardilhó,2024-09-15 10:53:00+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.00000000,2024-09-16 07:19:52.401000+00:00,"MULTIPOLYGON (((-8.60217 40.80174, -8.60225 40...",2024
57809,217103,ES,Asturias,Somiedo,2023-07-13 23:47:00+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00000000,2023-07-17 06:55:05.407000+00:00,"MULTIPOLYGON (((-6.26428 43.03166, -6.26433 43...",2023
28995,257399,EL,"Άνδρος, Θήρα, Κέα, Μήλος, Μύκονος, Νάξος, Πάρο...",Τοπική Κοινότητα Φαλατάδου,2025-02-14 10:08:00+00:00,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00000000,2025-03-06 09:45:47.310000+00:00,"MULTIPOLYGON (((25.20693 37.62229, 25.20686 37...",2025


## Create month column 

In [12]:
df_uniques.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 39782 entries, 6062 to 87991
Data columns (total 19 columns):
 #   Column      Non-Null Count  Dtype              
---  ------      --------------  -----              
 0   id          39782 non-null  int32              
 1   country     39782 non-null  str                
 2   province    39782 non-null  str                
 3   commune     39782 non-null  str                
 4   firedate    39782 non-null  datetime64[ms, UTC]
 5   area_ha     39782 non-null  int32              
 6   broadlea    39195 non-null  str                
 7   conifer     39195 non-null  str                
 8   mixed       39195 non-null  str                
 9   scleroph    39195 non-null  str                
 10  transit     39195 non-null  str                
 11  othernatlc  39195 non-null  str                
 12  agriareas   39195 non-null  str                
 13  artifsurf   39195 non-null  str                
 14  otherlc     39195 non-null  str 

In [15]:
df_uniques['month'] = df_uniques['firedate'].dt.strftime('%B')
df_uniques

,id,country,province,commune,firedate,area_ha,broadlea,conifer,mixed,scleroph,transit,othernatlc,agriareas,artifsurf,otherlc,percna2k,lastupdate,geometry,year,month
6062,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33333333,0.00000000,0.00000000,0.00000000,66.66666666,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:26:03.984000+00:00,"MULTIPOLYGON (((14.94778 40.74932, 14.94759 40...",2025,December
6063,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:25:40.928000+00:00,"MULTIPOLYGON (((14.93568 40.75572, 14.93565 40...",2025,December
6031,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00000000,2026-01-07 13:42:42.695000+00:00,"MULTIPOLYGON (((-8.38283 41.91932, -8.38284 41...",2025,December
6027,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:50:15.681000+00:00,"MULTIPOLYGON (((2.60406 45.06376, 2.60415 45.0...",2025,December
6044,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.66666667,0.00000000,0.00000000,0.00000000,0.00000000,83.33333333,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:21:16.864000+00:00,"MULTIPOLYGON (((2.60793 45.02059, 2.60768 45.0...",2025,December
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.11731844,0.00000000,0.00000000,0.00000000,26.25698324,58.10055866,14.52513966,0.00000000,0.00000000,99.46428403,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.82441 41.86385, -7.8275 41....",2015,January
92858,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77358491,0.00000000,0.00000000,0.00000000,0.00000000,96.22641509,0.00000000,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.79748 41.86824, -7.79693 41...",2015,January
86634,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00000000,0.00000000,0.00000000,0.00000000,44.01913876,53.11004785,2.87081340,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.90157 41.85033, -7.90371 41...",2015,January
92859,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,86.79245283,13.20754717,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.84382 41.84751, -7.84421 41...",2015,January


In [20]:
df_uniques.groupby('month')['area_ha'].sum().sort_values(ascending=False)

month
August       1905700
July         1157945
March         508906
October       474403
September     452514
February      359616
June          307199
April         168209
January       130230
May            68437
November       35666
December       25385
Name: area_ha, dtype: int32

## rename columns

In [21]:
rename_col = {
    "broadlea": "broadleaved_forest_pct",
    "conifer": "coniferous_forest_pct",
    "mixed": "mixed_forest_pct",
    "scleroph": "sclerophyllous_vegetation_pct",
    "transit": "transitional_woodland_shrub_pct",
    "othernatlc": "other_natural_land_cover_pct",
    "agriareas": "agricultural_areas_pct",
    "artifsurf": "artificial_surfaces_pct",
    "otherlc": "other_land_cover_pct",
    "percna2k": "natura2000_pct"
}

df_uniques_rename = df_uniques.rename(columns=rename_col)
df_uniques_rename

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,transitional_woodland_shrub_pct,other_natural_land_cover_pct,agricultural_areas_pct,artificial_surfaces_pct,other_land_cover_pct,natura2000_pct,lastupdate,geometry,year,month
6062,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33333333,0.00000000,0.00000000,0.00000000,66.66666666,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:26:03.984000+00:00,"MULTIPOLYGON (((14.94778 40.74932, 14.94759 40...",2025,December
6063,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,2026-01-07 12:25:40.928000+00:00,"MULTIPOLYGON (((14.93568 40.75572, 14.93565 40...",2025,December
6031,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00000000,2026-01-07 13:42:42.695000+00:00,"MULTIPOLYGON (((-8.38283 41.91932, -8.38284 41...",2025,December
6027,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,100.00000000,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:50:15.681000+00:00,"MULTIPOLYGON (((2.60406 45.06376, 2.60415 45.0...",2025,December
6044,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.66666667,0.00000000,0.00000000,0.00000000,0.00000000,83.33333333,0.00000000,0.00000000,0.00000000,0.00000000,2026-01-07 13:21:16.864000+00:00,"MULTIPOLYGON (((2.60793 45.02059, 2.60768 45.0...",2025,December
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.11731844,0.00000000,0.00000000,0.00000000,26.25698324,58.10055866,14.52513966,0.00000000,0.00000000,99.46428403,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.82441 41.86385, -7.8275 41....",2015,January
92858,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77358491,0.00000000,0.00000000,0.00000000,0.00000000,96.22641509,0.00000000,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.79748 41.86824, -7.79693 41...",2015,January
86634,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00000000,0.00000000,0.00000000,0.00000000,44.01913876,53.11004785,2.87081340,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.90157 41.85033, -7.90371 41...",2015,January
92859,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00000000,0.00000000,0.00000000,0.00000000,0.00000000,86.79245283,13.20754717,0.00000000,0.00000000,100.00000000,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((-7.84382 41.84751, -7.84421 41...",2015,January


## Turn % to ha 

In [22]:
pct_cols = [
    "broadleaved_forest_pct", #forest
    "coniferous_forest_pct", #forest
    "mixed_forest_pct", #forest
    "sclerophyllous_vegetation_pct", #natural ecosystem 
    "transitional_woodland_shrub_pct", #natural ecosystem
    "other_natural_land_cover_pct", #natural eco
    "agricultural_areas_pct", #agricultural areas
    "artificial_surfaces_pct", #civilian/industrial
    "other_land_cover_pct",
    "natura2000_pct" #narura 2000 protected areas
]

for col in pct_cols:
    df_uniques_rename[col] = pd.to_numeric(df_uniques_rename[col], errors="coerce")

for col in pct_cols:
    ha_col = col.replace("_pct", "_ha")
    df_uniques_rename[ha_col] = df_uniques_rename["area_ha"] * df_uniques_rename[col] / 100

df_uniques_rename

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,broadleaved_forest_ha,coniferous_forest_ha,mixed_forest_ha,sclerophyllous_vegetation_ha,transitional_woodland_shrub_ha,other_natural_land_cover_ha,agricultural_areas_ha,artificial_surfaces_ha,other_land_cover_ha,natura2000_ha
6062,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,...,0.67,0.00,0.00,0.00,1.33,0.00,0.00,0.00,0.00,2.00
6063,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,2.00
6031,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00
6027,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,5.00,0.00,0.00,0.00,0.00
6044,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,...,1.50,0.00,0.00,0.00,0.00,7.50,0.00,0.00,0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,...,2.00,0.00,0.00,0.00,47.00,104.00,26.00,0.00,0.00,178.04
92858,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,...,1.92,0.00,0.00,0.00,0.00,49.08,0.00,0.00,0.00,51.00
86634,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,92.44,111.53,6.03,0.00,0.00,210.00
92859,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.00,44.26,6.74,0.00,0.00,51.00


### create bigger categories

In [23]:
df_uniques_rename["forests_ha"] = (
    df_uniques_rename["broadleaved_forest_ha"] +
    df_uniques_rename["coniferous_forest_ha"] +
    df_uniques_rename["mixed_forest_ha"]
)

df_uniques_rename["natural_vegetation_ha"] = (
    df_uniques_rename["sclerophyllous_vegetation_ha"] +
    df_uniques_rename["transitional_woodland_shrub_ha"] +
    df_uniques_rename["other_natural_land_cover_ha"]
)

df_uniques_rename

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,mixed_forest_ha,sclerophyllous_vegetation_ha,transitional_woodland_shrub_ha,other_natural_land_cover_ha,agricultural_areas_ha,artificial_surfaces_ha,other_land_cover_ha,natura2000_ha,forests_ha,natural_vegetation_ha
6062,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,...,0.00,0.00,1.33,0.00,0.00,0.00,0.00,2.00,0.67,1.33
6063,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,...,0.00,0.00,2.00,0.00,0.00,0.00,0.00,2.00,0.00,2.00
6031,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN
6027,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,5.00,0.00,0.00,0.00,0.00,0.00,5.00
6044,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,...,0.00,0.00,0.00,7.50,0.00,0.00,0.00,0.00,1.50,7.50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87241,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,...,0.00,0.00,47.00,104.00,26.00,0.00,0.00,178.04,2.00,151.00
92858,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,...,0.00,0.00,0.00,49.08,0.00,0.00,0.00,51.00,1.92,49.08
86634,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,...,0.00,0.00,92.44,111.53,6.03,0.00,0.00,210.00,0.00,203.97
92859,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,44.26,6.74,0.00,0.00,51.00,0.00,44.26


### now do the overlap

In [24]:
nuts = gpd.read_file("NUTS_RG_01M_2024_3035.geojson")
nuts

,NUTS_ID,LEVL_CODE,CNTR_CODE,NAME_LATN,NUTS_NAME,MOUNT_TYPE,URBN_TYPE,COAST_TYPE,NAME_ENGL,NAME_FREN,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry
0,AL011,3,AL,Dibër,Dibër,NaN,NaN,NaN,Albania,Albanie,ALB,UN Member State,Tirana,F,F,T,Albanien,"MULTIPOLYGON (((5179983.544 2144882.937, 51798..."
1,AL012,3,AL,Durrës,Durrës,NaN,NaN,NaN,Albania,Albanie,ALB,UN Member State,Tirana,F,F,T,Albanien,"MULTIPOLYGON (((5139659.722 2104822.953, 51406..."
2,AL013,3,AL,Kukës,Kukës,NaN,NaN,NaN,Albania,Albanie,ALB,UN Member State,Tirana,F,F,T,Albanien,"MULTIPOLYGON (((5152332.168 2213667.282, 51562..."
3,AL031,3,AL,Berat,Berat,NaN,NaN,NaN,Albania,Albanie,ALB,UN Member State,Tirana,F,F,T,Albanien,"MULTIPOLYGON (((5162428.402 2029789.314, 51625..."
4,AL032,3,AL,Fier,Fier,NaN,NaN,NaN,Albania,Albanie,ALB,UN Member State,Tirana,F,F,T,Albanien,"MULTIPOLYGON (((5131123.546 2047723.145, 51311..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1793,RO,0,RO,România,România,NaN,NaN,NaN,Romania,Roumanie,ROU,UN Member State,Bucharest,T,F,F,Rumänien,"MULTIPOLYGON (((5800983.439 2519706.982, 58008..."
1794,NO,0,NO,Norge,Norge,NaN,NaN,NaN,Norway,Norvège,NOR,UN Member State,Oslo,F,T,F,Norwegen,"MULTIPOLYGON (((4172470.027 3876242.609, 41725..."
1795,PL,0,PL,Polska,Polska,NaN,NaN,NaN,Poland,Pologne,POL,UN Member State,Warsaw,T,F,F,Polen,"MULTIPOLYGON (((4620634.792 3405668.164, 46203..."
1796,PT,0,PT,Portugal,Portugal,NaN,NaN,NaN,Portugal,Portugal,PRT,UN Member State,Lisbon,T,F,F,Portugal,"MULTIPOLYGON (((2828494.611 2296190.204, 28279..."


Χρησιμοποιείς την στήλη geometry, που είναι ήδη GeoPandas geometry (MultiPolygon στη δική σου περίπτωση).
Τα NUTS polygons είναι επίσης στη geometry στήλη του nuts.

In [25]:
df_uniques_rename.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [26]:
nuts.crs

<Projected CRS: EPSG:3035>
Name: ETRS89-extended / LAEA Europe
Axis Info [cartesian]:
- Y[north]: Northing (metre)
- X[east]: Easting (metre)
Area of Use:
- name: Europe - European Union (EU) countries and candidates. Europe - onshore and offshore: Albania; Andorra; Austria; Belgium; Bosnia and Herzegovina; Bulgaria; Croatia; Cyprus; Czechia; Denmark; Estonia; Faroe Islands; Finland; France; Germany; Gibraltar; Greece; Hungary; Iceland; Ireland; Italy; Kosovo; Latvia; Liechtenstein; Lithuania; Luxembourg; Malta; Monaco; Montenegro; Netherlands; North Macedonia; Norway including Svalbard and Jan Mayen; Poland; Portugal including Madeira and Azores; Romania; San Marino; Serbia; Slovakia; Slovenia; Spain including Canary Islands; Sweden; Switzerland; Türkiye (Turkey); United Kingdom (UK) including Channel Islands and Isle of Man; Vatican City State.
- bounds: (-35.58, 24.6, 44.83, 84.73)
Coordinate Operation:
- name: Europe Equal Area 2001
- method: Lambert Azimuthal Equal Area
Datum: Eur

## καντα να εχουν το ιδιο projection 

In [27]:
df_geo_3035 = df_uniques_rename.to_crs(nuts.crs)

## χώρισε τα nuts 1/2/3

In [28]:
nuts1 = nuts[nuts["LEVL_CODE"] == 1]
nuts2 = nuts[nuts["LEVL_CODE"] == 2]
nuts3 = nuts[nuts["LEVL_CODE"] == 3]

### κάνεις intersection/overlay 

In [29]:
df_geo_3035_unique = df_geo_3035.drop_duplicates(subset="id") #περιττό αλλα για ασφάλεια
df_geo_3035_unique['original_polygon'] = df_geo_3035_unique['geometry']

In [30]:
intersections_nuts3 = gpd.overlay(df_geo_3035_unique, nuts3, how="intersection")

### υπολογίζεις το εμβαδόν επικάλυψης

In [31]:
intersections_nuts3["area_m2"] = intersections_nuts3.geometry.area

In [32]:
intersections_nuts3["geometry_area_ha"] = intersections_nuts3["area_m2"] / 10_000
intersections_nuts3

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry,area_m2,geometry_area_ha
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,...,ITA,UN Member State,Rome,T,F,F,Italien,"POLYGON ((4740465.655 1974931.971, 4740475.36 ...",24098.59,2.41
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,...,ITA,UN Member State,Rome,T,F,F,Italien,"POLYGON ((4739386.265 1975553.91, 4739386.265 ...",20449.50,2.04
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2807561.642 2276465.415, 2807559.234...",10163.58,1.02
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,...,FRA,UN Member State,Paris,T,F,F,Frankreich,"POLYGON ((3738600.805 2467963.017, 3738610.123...",50406.23,5.04
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,...,FRA,UN Member State,Paris,T,F,F,Frankreich,"POLYGON ((3738449.637 2463154.592, 3738431.272...",89046.04,8.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41251,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2851285.561 2259401.466, 2851587.313...",1794535.07,179.45
41252,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2853448.121 2259133.241, 2853347.537...",512884.87,51.29
41253,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2844774.341 2259449.133, 2845109.741...",2096355.91,209.64
41254,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2849256.126 2258019.912, 2849454.09 ...",514461.59,51.45


### τσεκ τις φωτιές που εμπίμπτουν σε πολλά nuts3

In [33]:
intersections_nuts3['id'] = intersections_nuts3['id'].astype(str)
intersections_nuts3['id'].value_counts()

id
206388    4
206890    4
280759    3
278497    3
278477    3
         ..
9109      1
9195      1
10658     1
9363      1
9087      1
Name: count, Length: 39778, dtype: int64

In [34]:
intersections_nuts3[intersections_nuts3['id'] == '206890']

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry,area_m2,geometry_area_ha
20935,206890,IT,Gorizia,Doberdò del Lago/Doberdob,2022-07-19 08:04:00+00:00,613,67.10,9.89,0.00,0.00,...,ITA,UN Member State,Rome,T,F,F,Italien,"POLYGON ((4600017.926 2531421.665, 4600124.472...",5277231.98,527.72
20936,206890,IT,Gorizia,Doberdò del Lago/Doberdob,2022-07-19 08:04:00+00:00,613,67.10,9.89,0.00,0.00,...,ITA,UN Member State,Rome,T,F,F,Italien,"POLYGON ((4601423.491 2528037.094, 4601694.442...",846011.68,84.60
20937,206890,IT,Gorizia,Doberdò del Lago/Doberdob,2022-07-19 08:04:00+00:00,613,67.10,9.89,0.00,0.00,...,SVN,UN Member State,Ljubljana,T,F,F,Slowenien,"MULTIPOLYGON (((4600050.909 2531460.594, 46001...",5877.04,0.59
20938,206890,IT,Gorizia,Doberdò del Lago/Doberdob,2022-07-19 08:04:00+00:00,613,67.10,9.89,0.00,0.00,...,SVN,UN Member State,Ljubljana,T,F,F,Slowenien,"MULTIPOLYGON (((4600848.818 2528851.327, 46008...",5671.16,0.57


In [35]:
#είναι το area από το geometry τετραγωικά μέτρα; αν ταυτίζεται με το area_ha είναι οκ (τσεκ ένα ha είναι 10,000 m2)

total_m2 = 5277231.98 + 846011.68 + 5877.04 + 5671.16
total_ha = total_m2 / 10000
total_ha

613.479186

#### άρα το country και το Province αντιστοιχούν με την χώρα και το nuts3 με την μεγαλύτερη καμμένη εκταση τις περισσότερες φορές

## κάνε merge με το αρχικο effis dataset (df_geo_3035_unique) τα intersection nuts 3

In [36]:
intersections_nuts3

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,ISO3_CODE,SVRG_UN,CAPT,EU_STAT,EFTA_STAT,CC_STAT,NAME_GERM,geometry,area_m2,geometry_area_ha
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,...,ITA,UN Member State,Rome,T,F,F,Italien,"POLYGON ((4740465.655 1974931.971, 4740475.36 ...",24098.59,2.41
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,...,ITA,UN Member State,Rome,T,F,F,Italien,"POLYGON ((4739386.265 1975553.91, 4739386.265 ...",20449.50,2.04
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2807561.642 2276465.415, 2807559.234...",10163.58,1.02
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,...,FRA,UN Member State,Paris,T,F,F,Frankreich,"POLYGON ((3738600.805 2467963.017, 3738610.123...",50406.23,5.04
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,...,FRA,UN Member State,Paris,T,F,F,Frankreich,"POLYGON ((3738449.637 2463154.592, 3738431.272...",89046.04,8.90
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41251,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2851285.561 2259401.466, 2851587.313...",1794535.07,179.45
41252,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2853448.121 2259133.241, 2853347.537...",512884.87,51.29
41253,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2844774.341 2259449.133, 2845109.741...",2096355.91,209.64
41254,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,...,PRT,UN Member State,Lisbon,T,F,F,Portugal,"POLYGON ((2849256.126 2258019.912, 2849454.09 ...",514461.59,51.45


In [38]:
intersections_nuts3 = intersections_nuts3.rename(columns={
    "NUTS_ID": "nuts3_id",
    "NUTS_NAME": "nuts3_name",
    "NAME_ENGL": "nuts3_country_name",
    "URBN_TYPE": "nuts3_urban_type",
    "COAST_TYPE": "nuts3_coast_type",
    "MOUNT_TYPE": "nuts3_mount_type",
    "EU_STAT": "eu_member" 
})

### σημαντικές πληροφορίες από τις στήλες για την ανάλυση

MOUNT_TYPE (ορεινότητα)
| Value | Meaning       |
| ----- | ------------- |
| 1     | Mountain area |
| 2     | Intermediate  |
| 3     | Non-mountain  |

    
URBN_TYPE (βαθμός αστικοποίησης)
| Value | Meaning             |
| ----- | ------------------- |
| 1     | Predominantly urban |
| 2     | Intermediate        |
| 3     | Predominantly rural |


COAST_TYPE (παράκτια περιοχή)
| Value | Meaning     |
| ----- | ----------- |
| 1     | Coastal     |
| 2     | Non-coastal |


urban & coastal = https://ec.europa.eu/eurostat/web/nuts/territorial-typologies?
mountain (simplified in our data but here you can see eurostat typologies) = https://ec.europa.eu/eurostat/cache/RCI/

In [42]:
urban_map = {
    1: "urban",
    2: "intermediate",
    3: "rural"
}

mountain_map = {
    1: "mountain area",
    2: "intermediate",
    3: "non-mountain area"
}

coastal_map = {
    1: "coastal",
    2: "non-coastal"
}


intersections_nuts3["nuts3_urban_rural"] = intersections_nuts3["nuts3_urban_type"].map(urban_map)
intersections_nuts3["nuts3_mountain"] = intersections_nuts3["nuts3_mount_type"].map(mountain_map)
intersections_nuts3["nuts3_coastal"] = intersections_nuts3["nuts3_coast_type"].map(coastal_map)

In [43]:
intersections_nuts3["nuts2_id"] = intersections_nuts3["nuts3_id"].str[:4]
intersections_nuts3["nuts1_id"] = intersections_nuts3["nuts3_id"].str[:3]

In [44]:
nuts2_attrs = nuts2[[
    "NUTS_ID", "NUTS_NAME"]].rename(columns={
    "NUTS_ID": "nuts2_id",
    "NUTS_NAME": "nuts2_name"})

nuts1_attrs = nuts1[[
    "NUTS_ID", "NUTS_NAME"]].rename(columns={
    "NUTS_ID": "nuts1_id",
    "NUTS_NAME": "nuts1_name"})

In [45]:
intersections_nuts3_2 = intersections_nuts3.merge(nuts2_attrs, on="nuts2_id", how="left")
intersections_nuts3_2_1 = intersections_nuts3_2.merge(nuts1_attrs, on="nuts1_id", how="left")

In [46]:
intersections_nuts3_2_1

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,geometry,area_m2,geometry_area_ha,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_id,nuts1_id,nuts2_name,nuts1_name
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,...,"POLYGON ((4740465.655 1974931.971, 4740475.36 ...",24098.59,2.41,intermediate,mountain area,coastal,ITF3,ITF,Campania,Sud
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,...,"POLYGON ((4739386.265 1975553.91, 4739386.265 ...",20449.50,2.04,intermediate,mountain area,coastal,ITF3,ITF,Campania,Sud
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,...,"POLYGON ((2807561.642 2276465.415, 2807559.234...",10163.58,1.02,rural,mountain area,coastal,PT11,PT1,Norte,Continente
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,...,"POLYGON ((3738600.805 2467963.017, 3738610.123...",50406.23,5.04,rural,mountain area,NaN,FRK1,FRK,Auvergne,Auvergne-Rhône-Alpes
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,...,"POLYGON ((3738449.637 2463154.592, 3738431.272...",89046.04,8.90,rural,mountain area,NaN,FRK1,FRK,Auvergne,Auvergne-Rhône-Alpes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41251,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,...,"POLYGON ((2851285.561 2259401.466, 2851587.313...",1794535.07,179.45,rural,mountain area,NaN,PT11,PT1,Norte,Continente
41252,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,...,"POLYGON ((2853448.121 2259133.241, 2853347.537...",512884.87,51.29,rural,mountain area,NaN,PT11,PT1,Norte,Continente
41253,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,...,"POLYGON ((2844774.341 2259449.133, 2845109.741...",2096355.91,209.64,rural,mountain area,NaN,PT11,PT1,Norte,Continente
41254,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,...,"POLYGON ((2849256.126 2258019.912, 2849454.09 ...",514461.59,51.45,rural,mountain area,NaN,PT11,PT1,Norte,Continente


In [47]:
cols_to_drop = [
    "LEVL_CODE",
    "CNTR_CODE",
    "NAME_LATN",
    "NAME_FREN",
    "ISO3_CODE",
    "SVRG_UN",
    "CAPT",
    "EFTA_STAT",
    "CC_STAT",
    "NAME_GERM",
    "nuts2_id",
    "nuts1_id"
]

intersections_nuts_all = intersections_nuts3_2_1.drop(columns=cols_to_drop)
intersections_nuts_all

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,nuts3_country_name,eu_member,geometry,area_m2,geometry_area_ha,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_name,nuts1_name
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,...,Italy,T,"POLYGON ((4740465.655 1974931.971, 4740475.36 ...",24098.59,2.41,intermediate,mountain area,coastal,Campania,Sud
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,...,Italy,T,"POLYGON ((4739386.265 1975553.91, 4739386.265 ...",20449.50,2.04,intermediate,mountain area,coastal,Campania,Sud
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,...,Portugal,T,"POLYGON ((2807561.642 2276465.415, 2807559.234...",10163.58,1.02,rural,mountain area,coastal,Norte,Continente
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,...,France,T,"POLYGON ((3738600.805 2467963.017, 3738610.123...",50406.23,5.04,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,...,France,T,"POLYGON ((3738449.637 2463154.592, 3738431.272...",89046.04,8.90,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41251,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,...,Portugal,T,"POLYGON ((2851285.561 2259401.466, 2851587.313...",1794535.07,179.45,rural,mountain area,NaN,Norte,Continente
41252,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,...,Portugal,T,"POLYGON ((2853448.121 2259133.241, 2853347.537...",512884.87,51.29,rural,mountain area,NaN,Norte,Continente
41253,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,...,Portugal,T,"POLYGON ((2844774.341 2259449.133, 2845109.741...",2096355.91,209.64,rural,mountain area,NaN,Norte,Continente
41254,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,...,Portugal,T,"POLYGON ((2849256.126 2258019.912, 2849454.09 ...",514461.59,51.45,rural,mountain area,NaN,Norte,Continente


In [48]:
intersections_nuts_all['country'].value_counts()

country
IT    10735
ES     7675
PT     6789
FR     5143
RO     4429
EL     1691
BG     1115
HR      921
DE      636
IE      500
SE      415
FI      283
PL      192
CY      167
HU      166
DK       99
LV       54
BE       42
NL       42
LT       37
SI       36
EE       28
AT       25
SK       17
CZ       15
MT        4
Name: count, dtype: int64

In [49]:
intersections_nuts_all['id'] = intersections_nuts_all['id'].astype(str)
df_geo_3035_unique['id'] = df_geo_3035_unique['id'].astype(str)

In [50]:
intersections_nuts_all.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 41256 entries, 0 to 41255
Data columns (total 47 columns):
 #   Column                           Non-Null Count  Dtype              
---  ------                           --------------  -----              
 0   id                               41256 non-null  str                
 1   country                          41256 non-null  str                
 2   province                         41256 non-null  str                
 3   commune                          41256 non-null  str                
 4   firedate                         41256 non-null  datetime64[ms, UTC]
 5   area_ha                          41256 non-null  int32              
 6   broadleaved_forest_pct           40664 non-null  float64            
 7   coniferous_forest_pct            40664 non-null  float64            
 8   mixed_forest_pct                 40664 non-null  float64            
 9   sclerophyllous_vegetation_pct    40664 non-null  float64        

In [51]:
df_geo_nuts_all = df_geo_3035_unique.merge(
    intersections_nuts_all[['id', 'geometry', 'geometry_area_ha', 'nuts3_id', 'nuts3_name', 
                            'nuts3_urban_rural', 'nuts3_mountain', 'nuts3_coastal',
                            'nuts2_name', 'nuts1_name']],
    on="id",
    how="left"
)
df_geo_nuts_all

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,...,original_polygon,geometry_y,geometry_area_ha,nuts3_id,nuts3_name,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_name,nuts1_name
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,...,"MULTIPOLYGON (((4740449.907 1974958.106, 47404...","POLYGON ((4740465.655 1974931.971, 4740475.36 ...",2.41,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,...,"MULTIPOLYGON (((4739383.871 1975597.602, 47393...","POLYGON ((4739386.265 1975553.91, 4739386.265 ...",2.04,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,...,"MULTIPOLYGON (((2807563.459 2276469.957, 28075...","POLYGON ((2807561.642 2276465.415, 2807559.234...",1.02,PT111,Alto Minho,rural,mountain area,coastal,Norte,Continente
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,...,"MULTIPOLYGON (((3738596.572 2467988.417, 37386...","POLYGON ((3738600.805 2467963.017, 3738610.123...",5.04,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,...,"MULTIPOLYGON (((3738448.918 2463186.93, 373842...","POLYGON ((3738449.637 2463154.592, 3738431.272...",8.90,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41255,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,...,"MULTIPOLYGON (((2851201.741 2259367.938, 28509...","POLYGON ((2851285.561 2259401.466, 2851587.313...",179.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente
41256,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,...,"MULTIPOLYGON (((2853481.649 2259317.646, 28535...","POLYGON ((2853448.121 2259133.241, 2853347.537...",51.29,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente
41257,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,...,"MULTIPOLYGON (((2844647.408 2259415.284, 28444...","POLYGON ((2844774.341 2259449.133, 2845109.741...",209.64,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente
41258,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,...,"MULTIPOLYGON (((2849240.349 2257976.523, 28491...","POLYGON ((2849256.126 2258019.912, 2849454.09 ...",51.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente


In [52]:
pd.set_option('display.max_columns', None)
df_geo_nuts_all

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,transitional_woodland_shrub_pct,other_natural_land_cover_pct,agricultural_areas_pct,artificial_surfaces_pct,other_land_cover_pct,natura2000_pct,lastupdate,geometry_x,year,month,broadleaved_forest_ha,coniferous_forest_ha,mixed_forest_ha,sclerophyllous_vegetation_ha,transitional_woodland_shrub_ha,other_natural_land_cover_ha,agricultural_areas_ha,artificial_surfaces_ha,other_land_cover_ha,natura2000_ha,forests_ha,natural_vegetation_ha,original_polygon,geometry_y,geometry_area_ha,nuts3_id,nuts3_name,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_name,nuts1_name
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,66.67,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:26:03.984000+00:00,"MULTIPOLYGON (((4740449.907 1974958.106, 47404...",2025,December,0.67,0.00,0.00,0.00,1.33,0.00,0.00,0.00,0.00,2.00,0.67,1.33,"MULTIPOLYGON (((4740449.907 1974958.106, 47404...","POLYGON ((4740465.655 1974931.971, 4740475.36 ...",2.41,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:25:40.928000+00:00,"MULTIPOLYGON (((4739383.871 1975597.602, 47393...",2025,December,0.00,0.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,2.00,0.00,2.00,"MULTIPOLYGON (((4739383.871 1975597.602, 47393...","POLYGON ((4739386.265 1975553.91, 4739386.265 ...",2.04,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,2026-01-07 13:42:42.695000+00:00,"MULTIPOLYGON (((2807563.459 2276469.957, 28075...",2025,December,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,"MULTIPOLYGON (((2807563.459 2276469.957, 28075...","POLYGON ((2807561.642 2276465.415, 2807559.234...",1.02,PT111,Alto Minho,rural,mountain area,coastal,Norte,Continente
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,2026-01-07 13:50:15.681000+00:00,"MULTIPOLYGON (((3738596.572 2467988.417, 37386...",2025,December,0.00,0.00,0.00,0.00,0.00,5.00,0.00,0.00,0.00,0.00,0.00,5.00,"MULTIPOLYGON (((3738596.572 2467988.417, 37386...","POLYGON ((3738600.805 2467963.017, 3738610.123...",5.04,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,0.00,83.33,0.00,0.00,0.00,0.00,2026-01-07 13:21:16.864000+00:00,"MULTIPOLYGON (((3738448.918 2463186.93, 373842...",2025,December,1.50,0.00,0.00,0.00,0.00,7.50,0.00,0.00,0.00,0.00,1.50,7.50,"MULTIPOLYGON (((3738448.918 2463186.93, 373842...","POLYGON ((3738449.637 2463154.592, 3738431.272...",8.90,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41255,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,26.26,58.10,14.53,0.00,0.00,99.46,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((2851201.741 2259367.938, 28509...",2015,January,2.00,0.00,0.00,0.00,47.00,104.00,26.00,0.00,0.00,178.04,2.00,151.00,"MULTIPOLYGON (((2851201.741 2259367.938, 28509...","POLYGON ((2851285.561 2259401.466, 2851587.313...",179.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente
41256,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,0.00,96.23,0.00,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,"MULTIPOLYGON (((2853481.649 2259317.646, 28535...",2015,January,1.92,0.00,0.00,0.00,0.00,49.08,0.00,0.00,0.00,51.00,1.92,49.08,"MULTIPOLYGON (((2853481.649 2259317.646, 28535...","POLYGON ((2853448.121 2259133.241, 285

In [53]:
df_geo_nuts_all_new = df_geo_nuts_all.copy()
df_geo_nuts_all_new.drop(columns='geometry_x', inplace=True)
df_geo_nuts_all_new.rename(columns={"geometry_y": "geometry_per_nuts3"}, inplace=True)
df_geo_nuts_all_new.rename(columns={"original_polygon": "original_fire_polygon"}, inplace=True)
df_geo_nuts_all_new

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,transitional_woodland_shrub_pct,other_natural_land_cover_pct,agricultural_areas_pct,artificial_surfaces_pct,other_land_cover_pct,natura2000_pct,lastupdate,year,month,broadleaved_forest_ha,coniferous_forest_ha,mixed_forest_ha,sclerophyllous_vegetation_ha,transitional_woodland_shrub_ha,other_natural_land_cover_ha,agricultural_areas_ha,artificial_surfaces_ha,other_land_cover_ha,natura2000_ha,forests_ha,natural_vegetation_ha,original_fire_polygon,geometry_per_nuts3,geometry_area_ha,nuts3_id,nuts3_name,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_name,nuts1_name
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,66.67,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:26:03.984000+00:00,2025,December,0.67,0.00,0.00,0.00,1.33,0.00,0.00,0.00,0.00,2.00,0.67,1.33,"MULTIPOLYGON (((4740449.907 1974958.106, 47404...","POLYGON ((4740465.655 1974931.971, 4740475.36 ...",2.41,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:25:40.928000+00:00,2025,December,0.00,0.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,2.00,0.00,2.00,"MULTIPOLYGON (((4739383.871 1975597.602, 47393...","POLYGON ((4739386.265 1975553.91, 4739386.265 ...",2.04,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,2026-01-07 13:42:42.695000+00:00,2025,December,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,"MULTIPOLYGON (((2807563.459 2276469.957, 28075...","POLYGON ((2807561.642 2276465.415, 2807559.234...",1.02,PT111,Alto Minho,rural,mountain area,coastal,Norte,Continente
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,2026-01-07 13:50:15.681000+00:00,2025,December,0.00,0.00,0.00,0.00,0.00,5.00,0.00,0.00,0.00,0.00,0.00,5.00,"MULTIPOLYGON (((3738596.572 2467988.417, 37386...","POLYGON ((3738600.805 2467963.017, 3738610.123...",5.04,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,0.00,83.33,0.00,0.00,0.00,0.00,2026-01-07 13:21:16.864000+00:00,2025,December,1.50,0.00,0.00,0.00,0.00,7.50,0.00,0.00,0.00,0.00,1.50,7.50,"MULTIPOLYGON (((3738448.918 2463186.93, 373842...","POLYGON ((3738449.637 2463154.592, 3738431.272...",8.90,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41255,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,26.26,58.10,14.53,0.00,0.00,99.46,2022-01-26 10:57:54.973000+00:00,2015,January,2.00,0.00,0.00,0.00,47.00,104.00,26.00,0.00,0.00,178.04,2.00,151.00,"MULTIPOLYGON (((2851201.741 2259367.938, 28509...","POLYGON ((2851285.561 2259401.466, 2851587.313...",179.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente
41256,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,0.00,96.23,0.00,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,1.92,0.00,0.00,0.00,0.00,49.08,0.00,0.00,0.00,51.00,1.92,49.08,"MULTIPOLYGON (((2853481.649 2259317.646, 28535...","POLYGON ((2853448.121 2259133.241, 2853347.537...",51.29,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente
41257,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,44.02,53.11,2.87,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,0.00,0.00,0.00,0.00,92.44,111.53,6.03,0.00,0.00,210.00,0.00,203.97,"MULTIPOLYGON (((2844647.408 225941

## Keep the nuts that correspond to the country of the fire

A fire can spread into multiple NUTS 3 regions and sometimes even in different neighbouring countries. So we keep first the regions of the country and calculate how much it overlapped in each region. 

In [55]:
df_geo_nuts_all_new["nuts_country"] = df_geo_nuts_all_new["nuts3_id"].str[:2]

# Keep only rows where EFFIS country matches NUTS country
df_geo_nuts_all_new_clean = df_geo_nuts_all_new[
    df_geo_nuts_all_new["country"] == df_geo_nuts_all_new["nuts_country"]
].copy()

df_geo_nuts_all_new_clean

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,transitional_woodland_shrub_pct,other_natural_land_cover_pct,agricultural_areas_pct,artificial_surfaces_pct,other_land_cover_pct,natura2000_pct,lastupdate,year,month,broadleaved_forest_ha,coniferous_forest_ha,mixed_forest_ha,sclerophyllous_vegetation_ha,transitional_woodland_shrub_ha,other_natural_land_cover_ha,agricultural_areas_ha,artificial_surfaces_ha,other_land_cover_ha,natura2000_ha,forests_ha,natural_vegetation_ha,original_fire_polygon,geometry_per_nuts3,geometry_area_ha,nuts3_id,nuts3_name,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_name,nuts1_name,nuts_country
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,66.67,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:26:03.984000+00:00,2025,December,0.67,0.00,0.00,0.00,1.33,0.00,0.00,0.00,0.00,2.00,0.67,1.33,"MULTIPOLYGON (((4740449.907 1974958.106, 47404...","POLYGON ((4740465.655 1974931.971, 4740475.36 ...",2.41,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud,IT
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:25:40.928000+00:00,2025,December,0.00,0.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,2.00,0.00,2.00,"MULTIPOLYGON (((4739383.871 1975597.602, 47393...","POLYGON ((4739386.265 1975553.91, 4739386.265 ...",2.04,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud,IT
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,2026-01-07 13:42:42.695000+00:00,2025,December,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,"MULTIPOLYGON (((2807563.459 2276469.957, 28075...","POLYGON ((2807561.642 2276465.415, 2807559.234...",1.02,PT111,Alto Minho,rural,mountain area,coastal,Norte,Continente,PT
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,2026-01-07 13:50:15.681000+00:00,2025,December,0.00,0.00,0.00,0.00,0.00,5.00,0.00,0.00,0.00,0.00,0.00,5.00,"MULTIPOLYGON (((3738596.572 2467988.417, 37386...","POLYGON ((3738600.805 2467963.017, 3738610.123...",5.04,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes,FR
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,0.00,83.33,0.00,0.00,0.00,0.00,2026-01-07 13:21:16.864000+00:00,2025,December,1.50,0.00,0.00,0.00,0.00,7.50,0.00,0.00,0.00,0.00,1.50,7.50,"MULTIPOLYGON (((3738448.918 2463186.93, 373842...","POLYGON ((3738449.637 2463154.592, 3738431.272...",8.90,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes,FR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41255,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,26.26,58.10,14.53,0.00,0.00,99.46,2022-01-26 10:57:54.973000+00:00,2015,January,2.00,0.00,0.00,0.00,47.00,104.00,26.00,0.00,0.00,178.04,2.00,151.00,"MULTIPOLYGON (((2851201.741 2259367.938, 28509...","POLYGON ((2851285.561 2259401.466, 2851587.313...",179.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT
41256,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,0.00,96.23,0.00,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,1.92,0.00,0.00,0.00,0.00,49.08,0.00,0.00,0.00,51.00,1.92,49.08,"MULTIPOLYGON (((2853481.649 2259317.646, 28535...","POLYGON ((2853448.121 2259133.241, 2853347.537...",51.29,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT
41257,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,44.02,53.11,2.87,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,0.00,0.00,0.00,0.00,92.44,111.53,6.03,0.00,0.00,210.00,0.00,203.

## FIX the geometry overlay of each NUTS 3 

Normalise geometry areas, they are slightly bigger than the original fire grouped together

In [57]:
import numpy as np 

df_geo_nuts_all_normalised = df_geo_nuts_all_new_clean.copy()

# Ensure numeric
df_geo_nuts_all_normalised["geometry_area_ha"] = pd.to_numeric(
    df_geo_nuts_all_normalised["geometry_area_ha"],
    errors="coerce"
)

df_geo_nuts_all_normalised["area_ha"] = pd.to_numeric(
    df_geo_nuts_all_normalised["area_ha"],
    errors="coerce"
)

# --------------------------------------------------
# 2. Normalize NUTS3 intersection areas per fire
# so each fire's NUTS3 pieces sum back to real area_ha
# --------------------------------------------------

fire_totals = (
    df_geo_nuts_all_normalised
    .groupby("id", dropna=False)["geometry_area_ha"]
    .sum()
    .reset_index(name="total_geometry_area_ha")
)

df_geo_nuts_all_normalised = df_geo_nuts_all_normalised.merge(
    fire_totals,
    on="id",
    how="left"
)

df_geo_nuts_all_normalised["geometry_area_ha_normalized"] = np.where(
    df_geo_nuts_all_normalised["total_geometry_area_ha"] > 0,
    (df_geo_nuts_all_normalised["geometry_area_ha"] / df_geo_nuts_all_normalised["total_geometry_area_ha"]) * df_geo_nuts_all_normalised["area_ha"],
    0
) 

df_geo_nuts_all_normalised

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,transitional_woodland_shrub_pct,other_natural_land_cover_pct,agricultural_areas_pct,artificial_surfaces_pct,other_land_cover_pct,natura2000_pct,lastupdate,year,month,broadleaved_forest_ha,coniferous_forest_ha,mixed_forest_ha,sclerophyllous_vegetation_ha,transitional_woodland_shrub_ha,other_natural_land_cover_ha,agricultural_areas_ha,artificial_surfaces_ha,other_land_cover_ha,natura2000_ha,forests_ha,natural_vegetation_ha,original_fire_polygon,geometry_per_nuts3,geometry_area_ha,nuts3_id,nuts3_name,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_name,nuts1_name,nuts_country,total_geometry_area_ha,geometry_area_ha_normalized
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,66.67,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:26:03.984000+00:00,2025,December,0.67,0.00,0.00,0.00,1.33,0.00,0.00,0.00,0.00,2.00,0.67,1.33,"MULTIPOLYGON (((4740449.907 1974958.106, 47404...","POLYGON ((4740465.655 1974931.971, 4740475.36 ...",2.41,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud,IT,2.41,2.00
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:25:40.928000+00:00,2025,December,0.00,0.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,2.00,0.00,2.00,"MULTIPOLYGON (((4739383.871 1975597.602, 47393...","POLYGON ((4739386.265 1975553.91, 4739386.265 ...",2.04,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud,IT,2.04,2.00
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,2026-01-07 13:42:42.695000+00:00,2025,December,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,"MULTIPOLYGON (((2807563.459 2276469.957, 28075...","POLYGON ((2807561.642 2276465.415, 2807559.234...",1.02,PT111,Alto Minho,rural,mountain area,coastal,Norte,Continente,PT,1.02,1.00
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,2026-01-07 13:50:15.681000+00:00,2025,December,0.00,0.00,0.00,0.00,0.00,5.00,0.00,0.00,0.00,0.00,0.00,5.00,"MULTIPOLYGON (((3738596.572 2467988.417, 37386...","POLYGON ((3738600.805 2467963.017, 3738610.123...",5.04,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes,FR,5.04,5.00
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,0.00,83.33,0.00,0.00,0.00,0.00,2026-01-07 13:21:16.864000+00:00,2025,December,1.50,0.00,0.00,0.00,0.00,7.50,0.00,0.00,0.00,0.00,1.50,7.50,"MULTIPOLYGON (((3738448.918 2463186.93, 373842...","POLYGON ((3738449.637 2463154.592, 3738431.272...",8.90,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes,FR,8.90,9.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41050,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,26.26,58.10,14.53,0.00,0.00,99.46,2022-01-26 10:57:54.973000+00:00,2015,January,2.00,0.00,0.00,0.00,47.00,104.00,26.00,0.00,0.00,178.04,2.00,151.00,"MULTIPOLYGON (((2851201.741 2259367.938, 28509...","POLYGON ((2851285.561 2259401.466, 2851587.313...",179.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT,179.45,179.00
41051,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,0.00,96.23,0.00,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,1.92,0.00,0.00,0.00,0.00,49.08,0.00,0.00,0.00,51.00,1.92,49.08,"MULTIPOLYGON (((2853481.649 2259317.646, 28535...","POLYGON ((2853448.121 2259133.241, 2853347.537...",51.29,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT,51.29,51.00
41052,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,44.02,53.

### for viz

In [58]:
df_geo_nuts_all_new_for_analysis = df_geo_nuts_all_normalised.drop(
    columns=["original_fire_polygon", "geometry_per_nuts3"]
)
df_geo_nuts_all_new_for_analysis

,id,country,province,commune,firedate,area_ha,broadleaved_forest_pct,coniferous_forest_pct,mixed_forest_pct,sclerophyllous_vegetation_pct,transitional_woodland_shrub_pct,other_natural_land_cover_pct,agricultural_areas_pct,artificial_surfaces_pct,other_land_cover_pct,natura2000_pct,lastupdate,year,month,broadleaved_forest_ha,coniferous_forest_ha,mixed_forest_ha,sclerophyllous_vegetation_ha,transitional_woodland_shrub_ha,other_natural_land_cover_ha,agricultural_areas_ha,artificial_surfaces_ha,other_land_cover_ha,natura2000_ha,forests_ha,natural_vegetation_ha,geometry_area_ha,nuts3_id,nuts3_name,nuts3_urban_rural,nuts3_mountain,nuts3_coastal,nuts2_name,nuts1_name,nuts_country,total_geometry_area_ha,geometry_area_ha_normalized
0,288077,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,33.33,0.00,0.00,0.00,66.67,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:26:03.984000+00:00,2025,December,0.67,0.00,0.00,0.00,1.33,0.00,0.00,0.00,0.00,2.00,0.67,1.33,2.41,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud,IT,2.41,2.00
1,288076,IT,Salerno,Giffoni Valle Piana,2025-12-31 23:00:00+00:00,2,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,100.00,2026-01-07 12:25:40.928000+00:00,2025,December,0.00,0.00,0.00,0.00,2.00,0.00,0.00,0.00,0.00,2.00,0.00,2.00,2.04,ITF35,Salerno,intermediate,mountain area,coastal,Campania,Sud,IT,2.04,2.00
2,288108,PT,Alto Minho,Gondoriz,2025-12-31 13:12:00+00:00,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100.00,2026-01-07 13:42:42.695000+00:00,2025,December,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.00,NaN,NaN,1.02,PT111,Alto Minho,rural,mountain area,coastal,Norte,Continente,PT,1.02,1.00
3,288112,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 12:44:00+00:00,5,0.00,0.00,0.00,0.00,0.00,100.00,0.00,0.00,0.00,0.00,2026-01-07 13:50:15.681000+00:00,2025,December,0.00,0.00,0.00,0.00,0.00,5.00,0.00,0.00,0.00,0.00,0.00,5.00,5.04,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes,FR,5.04,5.00
4,288095,FR,Cantal,Saint-Cirgues-de-Jordanne,2025-12-31 11:31:00+00:00,9,16.67,0.00,0.00,0.00,0.00,83.33,0.00,0.00,0.00,0.00,2026-01-07 13:21:16.864000+00:00,2025,December,1.50,0.00,0.00,0.00,0.00,7.50,0.00,0.00,0.00,0.00,1.50,7.50,8.90,FRK12,Cantal,rural,mountain area,NaN,Auvergne,Auvergne-Rhône-Alpes,FR,8.90,9.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41050,9109,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-24 23:00:00+00:00,179,1.12,0.00,0.00,0.00,26.26,58.10,14.53,0.00,0.00,99.46,2022-01-26 10:57:54.973000+00:00,2015,January,2.00,0.00,0.00,0.00,47.00,104.00,26.00,0.00,0.00,178.04,2.00,151.00,179.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT,179.45,179.00
41051,9195,PT,Alto Tâmega,Montalegre e Padroso,2015-01-24 23:00:00+00:00,51,3.77,0.00,0.00,0.00,0.00,96.23,0.00,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,1.92,0.00,0.00,0.00,0.00,49.08,0.00,0.00,0.00,51.00,1.92,49.08,51.29,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT,51.29,51.00
41052,10658,PT,Alto Tâmega,Sezelhe e Covelães,2015-01-24 23:00:00+00:00,210,0.00,0.00,0.00,0.00,44.02,53.11,2.87,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,0.00,0.00,0.00,0.00,92.44,111.53,6.03,0.00,0.00,210.00,0.00,203.97,209.64,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT,209.64,210.00
41053,9363,PT,Alto Tâmega,"Cambeses do Rio, Donões e Mourilhe",2015-01-14 23:00:00+00:00,51,0.00,0.00,0.00,0.00,0.00,86.79,13.21,0.00,0.00,100.00,2022-01-26 10:57:54.973000+00:00,2015,January,0.00,0.00,0.00,0.00,0.00,44.26,6.74,0.00,0.00,51.00,0.00,44.26,51.45,PT11B,Alto Tâmega e Barroso,rural,mountain area,NaN,Norte,Continente,PT,51.45,51.00


In [59]:
df_geo_nuts_all_normalised = df_geo_nuts_all_normalised.set_geometry("geometry_per_nuts3")
df_geo_nuts_all_new_for_viz = df_geo_nuts_all_normalised.drop(columns="original_fire_polygon")

df_geo_nuts_all_new_for_viz.crs

<Projected CRS: EPSG:3035>
Name: ETRS89-extended / LAEA Europe
Axis Info [cartesian]:
- Y[north]: Northing (metre)
- X[east]: Easting (metre)
Area of Use:
- name: Europe - European Union (EU) countries and candidates. Europe - onshore and offshore: Albania; Andorra; Austria; Belgium; Bosnia and Herzegovina; Bulgaria; Croatia; Cyprus; Czechia; Denmark; Estonia; Faroe Islands; Finland; France; Germany; Gibraltar; Greece; Hungary; Iceland; Ireland; Italy; Kosovo; Latvia; Liechtenstein; Lithuania; Luxembourg; Malta; Monaco; Montenegro; Netherlands; North Macedonia; Norway including Svalbard and Jan Mayen; Poland; Portugal including Madeira and Azores; Romania; San Marino; Serbia; Slovakia; Slovenia; Spain including Canary Islands; Sweden; Switzerland; Türkiye (Turkey); United Kingdom (UK) including Channel Islands and Isle of Man; Vatican City State.
- bounds: (-35.58, 24.6, 44.83, 84.73)
Coordinate Operation:
- name: Europe Equal Area 2001
- method: Lambert Azimuthal Equal Area
Datum: Eur

In [60]:
df_geo_nuts_all_mapbox = df_geo_nuts_all_new_for_viz.to_crs(4326)
df_geo_nuts_all_mapbox.to_file("forest_fires_nutsall_mapbox_4326.geojson", driver='GeoJSON')

## for analysis with pandas

In [61]:
df_geo_nuts_all_new_for_analysis.to_csv("forest_fires_for_analysis_all_nuts_normalised_area.csv", index=False)

# Links 

nuts from here: https://ec.europa.eu/eurostat/web/gisco/geodata/statistical-units/territorial-units-statistics

date of spatial join 4 april 2026

## check the romania fires of the danube delta 

In [66]:
fires_ro = [
"10264", "11534", "11605", "12190", "12470", "12776", "12794",
"12969", "15213", "15674", "15692", "15747", "15788", "15971",
"16050", "16093", "16426", "17659", "17887", "180442", "181344",
"183648", "183649", "183657", "183664", "190199", "19156", "192631",
"19305", "19311", "19371", "19468", "19545", "19614", "19750", "198984",
"19915", "200517", "200533", "211164", "211167", "211168", "211810",
"211812", "211815", "213342", "213574", "216991", "22098", "22116",
"22294", "234328", "23998", "24412", "256334", "258178", "258254", "258523",
"258530", "265255", "265521", "265564", "265593", "267173", "267174", "268177",
"268238", "268239", "270324", "281024", "43174", "45031", "45053", "46395", "46765",
"47173", "47724", "59220", "9218"
]

romania_fires_delta = df_geo_nuts_all_mapbox[
    df_geo_nuts_all_mapbox["id"].isin(fires_ro)
].copy()

romania_fires_delta_3035 = romania_fires_delta.to_crs(3035)
romania_fires_delta_3035.to_file(
    "romania_fires_delta.gpkg",
    driver="GPKG"
)